# S3 — Permutation Test for Relationship Detection

For each of the 114,176 synthetic cases, tests whether x and y have a
detectable relationship by comparing four observed metrics against their
permutation null distributions (shuffle y, keep x fixed).

**Metrics (mutually complementary):**

| Metric | Captures |
|---|---|
| \|Pearson r\| | Linear dependence |
| \|Spearman ρ\| | Monotonic dependence |
| Distance correlation | Arbitrary dependence |
| η² (equal-width bins) | Mean-response signal across x-regions |

**Computation split into 3 phases (crash-safe):**

| Phase | Content | Speed | Checkpoint |
|---|---|---|---|
| Phase 1 | \|Pearson\|, \|Spearman\|, η² | ~20 min (vectorised) | `_perm_phase1.npz` |
| Phase 2 | Distance correlation | ~4-6 h (loop) | `_perm_phase2.npz` |
| Phase 3 | Z-scores + joint test + classify | ~1 min | final parquet |

**Input:** `generated_scatterplot_data/cases.csv` + `scatter_points.npz`

**Output:** `generated_scatterplot_data/full/S3/permutation_test.parquet`

In [1]:
from __future__ import annotations

import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from scipy.stats import rankdata

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kw):
        return it

warnings.filterwarnings("ignore")

DATA_DIR  = Path("generated_scatterplot_data")
OUT_DIR   = DATA_DIR / "full" / "S3"
OUT_DIR.mkdir(parents=True, exist_ok=True)
N_PERM    = 500
SEED_BASE = 42_000_000

/Users/mimi/miniconda3/envs/pip39/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data

In [2]:
cases_df = pd.read_csv(DATA_DIR / "cases.csv", low_memory=False)
data     = np.load(DATA_DIR / "scatter_points.npz")
x_all    = data["x"]
y_all    = data["y"]
n_cases, n_points = x_all.shape
print(f"Loaded {n_cases:,} cases × {n_points} points")
print(f"Null={int((cases_df.family_id == 'Null').sum())}, Signal={int((cases_df.family_id != 'Null').sum())}")

Loaded 114,176 cases × 500 points
Null=128, Signal=114048


## Helper Functions

In [3]:
def _double_center(a):
    D = squareform(pdist(a.reshape(-1, 1)))
    return D - D.mean(axis=0, keepdims=True) - D.mean(axis=1, keepdims=True) + D.mean()


def _precompute_bins(x, n_bins=10, min_count=5):
    try:
        bins = np.asarray(
            pd.cut(x, bins=n_bins, labels=False, include_lowest=True, duplicates="drop"),
            dtype=float,
        )
    except Exception:
        return None
    valid_ids = np.unique(bins[~np.isnan(bins)]).astype(int)
    masks, counts = [], []
    for b in valid_ids:
        m = bins == b
        if m.sum() >= min_count:
            masks.append(m)
            counts.append(m.sum())
    if len(masks) < 2:
        return None
    B_ind = np.array([m.astype(np.float64) for m in masks])
    return B_ind, np.array(counts, dtype=np.float64)


def _generate_perms(n, n_perm, seed):
    rng = np.random.default_rng(seed)
    return np.array([rng.permutation(n) for _ in range(n_perm)])

## Phase 1 — |Pearson|, |Spearman|, η²

All three are **vectorised** across 500 permutations per case.
Estimated time: ~20 minutes. Saves checkpoint to `_perm_phase1.npz`.

In [4]:
pearson_obs_all  = np.empty(n_cases)
spearman_obs_all = np.empty(n_cases)
eta2_obs_all     = np.empty(n_cases)

pearson_null_all  = np.empty((n_cases, N_PERM), dtype=np.float32)
spearman_null_all = np.empty((n_cases, N_PERM), dtype=np.float32)
eta2_null_all     = np.empty((n_cases, N_PERM), dtype=np.float32)

t0 = time.time()

for i in tqdm(range(n_cases), desc="Phase 1"):
    x = x_all[i].astype(np.float64)
    y = y_all[i].astype(np.float64)
    perms = _generate_perms(len(x), N_PERM, SEED_BASE + i)

    # ── |Pearson| ──
    xc = x - x.mean()
    yc = y - y.mean()
    sx = np.sqrt((xc ** 2).sum())
    sy = np.sqrt((yc ** 2).sum())
    if sx > 0 and sy > 0:
        denom = sx * sy
        pearson_obs_all[i]  = abs(float((xc * yc).sum() / denom))
        pearson_null_all[i] = np.abs((xc * yc[perms]).sum(axis=1) / denom)
    else:
        pearson_obs_all[i]  = 0.0
        pearson_null_all[i] = 0.0

    # ── |Spearman| ──
    xr  = rankdata(x).astype(np.float64)
    yr  = rankdata(y).astype(np.float64)
    xrc = xr - xr.mean()
    yrc = yr - yr.mean()
    sxr = np.sqrt((xrc ** 2).sum())
    syr = np.sqrt((yrc ** 2).sum())
    if sxr > 0 and syr > 0:
        denom_r = sxr * syr
        spearman_obs_all[i]  = abs(float((xrc * yrc).sum() / denom_r))
        spearman_null_all[i] = np.abs((xrc * yrc[perms]).sum(axis=1) / denom_r)
    else:
        spearman_obs_all[i]  = 0.0
        spearman_null_all[i] = 0.0

    # ── η² ──
    bin_info = _precompute_bins(x)
    y_mean   = float(y.mean())
    ss_total = float(((y - y_mean) ** 2).sum())
    if bin_info is not None and ss_total > 0:
        B_ind, bin_counts = bin_info
        bin_means_obs = (B_ind @ y) / bin_counts
        eta2_obs_all[i] = float((bin_counts * (bin_means_obs - y_mean) ** 2).sum() / ss_total)
        y_perms = y[perms]
        bin_means_null = (y_perms @ B_ind.T) / bin_counts
        ss_bet_null = (bin_counts * (bin_means_null - y_mean) ** 2).sum(axis=1)
        eta2_null_all[i] = (ss_bet_null / ss_total).astype(np.float32)
    else:
        eta2_obs_all[i]  = 0.0
        eta2_null_all[i] = 0.0

elapsed = time.time() - t0
print(f"\nPhase 1 done: {n_cases:,} cases in {elapsed/60:.1f} min ({n_cases/elapsed:.0f} cases/s)")

# Checkpoint
ckpt1 = OUT_DIR / "_perm_phase1.npz"
np.savez_compressed(ckpt1,
    pearson_obs=pearson_obs_all, spearman_obs=spearman_obs_all, eta2_obs=eta2_obs_all,
    pearson_null=pearson_null_all, spearman_null=spearman_null_all, eta2_null=eta2_null_all,
)
print(f"Checkpoint saved: {ckpt1}  ({ckpt1.stat().st_size / 1e6:.0f} MB)")

Phase 1: 100%|██████████| 114176/114176 [18:20<00:00, 103.79it/s]



Phase 1 done: 114,176 cases in 18.3 min (104 cases/s)
Checkpoint saved: generated_scatterplot_data/full/S3/_perm_phase1.npz  (615 MB)


## Phase 2 — Distance Correlation

Main bottleneck: 500×500 distance matrix permutation loop per case.
Estimated time: ~4-6 hours. Saves checkpoint to `_perm_phase2.npz`.

In [6]:
dcor_obs_all  = np.empty(n_cases)
dcor_null_all = np.empty((n_cases, N_PERM), dtype=np.float32)

t0 = time.time()

for i in tqdm(range(n_cases), desc="Phase 2"):
    x = x_all[i].astype(np.float64)
    y = y_all[i].astype(np.float64)
    perms = _generate_perms(len(x), N_PERM, SEED_BASE + i)

    A = _double_center(x)
    B = _double_center(y)
    dcov_xx = np.sqrt(max((A * A).mean(), 0))
    dcov_yy = np.sqrt(max((B * B).mean(), 0))
    dcov_denom = np.sqrt(dcov_xx * dcov_yy)

    # Observed
    dcov_obs = np.sqrt(max(float((A * B).mean()), 0))
    dcor_obs_all[i] = float(dcov_obs / dcov_denom) if dcov_denom > 0 else 0.0

    # Null (loop)
    if dcov_denom > 0:
        for k in range(N_PERM):
            p = perms[k]
            dcov2 = float((A * B[p][:, p]).mean())
            dcor_null_all[i, k] = np.sqrt(max(dcov2, 0)) / dcov_denom
    else:
        dcor_null_all[i] = 0.0

    if (i + 1) % 5000 == 0:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        eta_h = (n_cases - i - 1) / rate / 3600
        print(f"  {i+1:>7,}/{n_cases:,}  ({rate:.1f} cases/s, ETA {eta_h:.1f} h)")

elapsed = time.time() - t0
print(f"\nPhase 2 done: {n_cases:,} cases in {elapsed/3600:.1f} hours ({n_cases/elapsed:.1f} cases/s)")

# Checkpoint
ckpt2 = OUT_DIR / "_perm_phase2.npz"
np.savez_compressed(ckpt2, dcor_obs=dcor_obs_all, dcor_null=dcor_null_all)
print(f"Checkpoint saved: {ckpt2}  ({ckpt2.stat().st_size / 1e6:.0f} MB)")

Phase 2:   4%|▍         | 5000/114176 [36:30<14:14:02,  2.13it/s]

    5,000/114,176  (2.3 cases/s, ETA 13.3 h)


Phase 2:   9%|▉         | 10000/114176 [1:13:26<12:31:14,  2.31it/s]

   10,000/114,176  (2.3 cases/s, ETA 12.8 h)


Phase 2:  13%|█▎        | 15000/114176 [1:50:39<12:55:34,  2.13it/s]

...

  105,000/114,176  (2.2 cases/s, ETA 1.2 h)


Phase 2:  96%|█████████▋| 110000/114176 [14:03:37<32:25,  2.15it/s]  

  110,000/114,176  (2.2 cases/s, ETA 0.5 h)


Phase 2: 100%|██████████| 114176/114176 [14:35:45<00:00,  2.17it/s]



Phase 2 done: 114,176 cases in 14.6 hours (2.2 cases/s)
Checkpoint saved: generated_scatterplot_data/full/S3/_perm_phase2.npz  (196 MB)


## Phase 3 — Z-scores, Joint Test & Classification

Loads Phase 1 + Phase 2 results (from memory or checkpoint files),
computes Z-scores, joint T = max(Z), and p-values.

In [7]:
# Load from memory; fall back to checkpoint files if kernel was restarted
try:
    _ = pearson_obs_all.shape
    print("Phase 1 data in memory ✓")
except NameError:
    print("Loading Phase 1 from checkpoint...")
    p1 = np.load(OUT_DIR / "_perm_phase1.npz")
    pearson_obs_all   = p1["pearson_obs"]
    spearman_obs_all  = p1["spearman_obs"]
    eta2_obs_all      = p1["eta2_obs"]
    pearson_null_all  = p1["pearson_null"]
    spearman_null_all = p1["spearman_null"]
    eta2_null_all     = p1["eta2_null"]
    print("  loaded ✓")

try:
    _ = dcor_obs_all.shape
    print("Phase 2 data in memory ✓")
except NameError:
    print("Loading Phase 2 from checkpoint...")
    p2 = np.load(OUT_DIR / "_perm_phase2.npz")
    dcor_obs_all  = p2["dcor_obs"]
    dcor_null_all = p2["dcor_null"]
    print("  loaded ✓")

print(f"Assembling results: {n_cases:,} cases × {N_PERM} permutations × 4 metrics")

# Compute Z-scores + joint test per case
names = ["pearson", "spearman", "dcor", "eta2"]
all_obs  = [pearson_obs_all, spearman_obs_all, dcor_obs_all, eta2_obs_all]
all_null = [pearson_null_all, spearman_null_all, dcor_null_all, eta2_null_all]

results = []
t0 = time.time()

for i in tqdm(range(n_cases), desc="Phase 3"):
    row = {}
    z_obs_list  = []
    z_null_list = []

    for name, obs_arr, null_arr in zip(names, all_obs, all_null):
        obs  = float(obs_arr[i])
        null = null_arr[i].astype(np.float64)

        med = float(np.median(null))
        iqr = float(np.percentile(null, 75) - np.percentile(null, 25))
        if iqr < 1e-12:
            iqr = float(np.std(null)) * 1.35
        if iqr < 1e-12:
            z_o, z_n = 0.0, np.zeros(N_PERM)
        else:
            z_o = (obs - med) / iqr
            z_n = (null - med) / iqr

        z_obs_list.append(z_o)
        z_null_list.append(z_n)
        row[f"{name}_obs"]         = obs
        row[f"{name}_null_median"] = med
        row[f"{name}_null_iqr"]    = iqr
        row[f"z_{name}"]           = float(z_o)

    T_obs   = max(z_obs_list)
    T_null  = np.stack(z_null_list).max(axis=0)
    p_value = float(np.sum(T_null >= T_obs) + 1) / (N_PERM + 1)

    row["T_joint"] = float(T_obs)
    row["p_value"] = float(p_value)
    results.append(row)

elapsed = time.time() - t0
print(f"\nPhase 3 done in {elapsed:.0f}s")

Phase 1 data in memory ✓
Phase 2 data in memory ✓
Assembling results: 114,176 cases × 500 permutations × 4 metrics


Phase 3: 100%|██████████| 114176/114176 [00:47<00:00, 2413.67it/s]


Phase 3 done in 47s


## Classification & Save

| p-value | Classification |
|---|---|
| p ≤ 0.05 | Detectable |
| 0.05 < p < 0.10 | Uncertain |
| p ≥ 0.10 | Not detectable |

In [8]:
perm_df = pd.DataFrame(results)
perm_df.insert(0, "case_id", cases_df["case_id"].values)

# Column ordering
col_order = [
    "case_id",
    "pearson_obs", "spearman_obs", "dcor_obs", "eta2_obs",
    "pearson_null_median", "pearson_null_iqr",
    "spearman_null_median", "spearman_null_iqr",
    "dcor_null_median", "dcor_null_iqr",
    "eta2_null_median", "eta2_null_iqr",
    "z_pearson", "z_spearman", "z_dcor", "z_eta2",
    "T_joint", "p_value",
]
perm_df = perm_df[[c for c in col_order if c in perm_df.columns]]

# Classify
perm_df["classification"] = "uncertain"
perm_df.loc[perm_df["p_value"] <= 0.05, "classification"] = "detectable"
perm_df.loc[perm_df["p_value"] >= 0.10, "classification"] = "not_detectable"

# Save
out_path = OUT_DIR / "permutation_test.parquet"
perm_df.to_parquet(out_path, index=False)
size_mb = out_path.stat().st_size / 1e6
print(f"Saved {out_path}  ({len(perm_df):,} rows × {len(perm_df.columns)} cols, {size_mb:.1f} MB)")
print()
print("Classification distribution:")
print(perm_df["classification"].value_counts().to_string())
print()
display(perm_df.head(10))

Saved generated_scatterplot_data/full/S3/permutation_test.parquet  (114,176 rows × 20 cols, 18.1 MB)

Classification distribution:
classification
detectable        113127
not_detectable       810
uncertain            239



,case_id,pearson_obs,spearman_obs,dcor_obs,eta2_obs,pearson_null_median,pearson_null_iqr,spearman_null_median,spearman_null_iqr,dcor_null_median,dcor_null_iqr,eta2_null_median,eta2_null_iqr,z_pearson,z_spearman,z_dcor,z_eta2,T_joint,p_value,classification
0,1,0.270654,0.262035,0.257996,0.093535,0.029195,0.037669,0.030172,0.037546,0.070004,0.020475,0.016468,0.011106,6.410016,6.175383,9.181658,6.939113,9.181658,0.001996,detectable
1,2,0.202792,0.197157,0.186921,0.056960,0.029574,0.036319,0.028551,0.034933,0.071731,0.018920,0.014766,0.010367,4.769383,4.826583,6.088212,4.069988,6.088212,0.001996,detectable
2,3,0.141068,0.133403,0.151419,0.024620,0.030216,0.035481,0.029896,0.035280,0.071490,0.019912,0.014952,0.009909,3.124254,2.933879,4.014150,0.975597,4.014150,0.001996,detectable
3,4,0.188816,0.161542,0.170316,0.058786,0.030557,0.037586,0.028793,0.038404,0.073238,0.019098,0.016665,0.011734,4.210549,3.456599,5.083292,3.589806,5.083292,0.001996,detectable
4,5,0.208436,0.200662,0.187598,0.055586,0.032402,0.036879,0.032206,0.038949,0.065504,0.020926,0.014285,0.009836,4.773286,4.325090,5.834440,4.199037,5.834440,0.001996,detectable
5,6,0.259839,0.274240,0.252238,0.069328,0.030814,0.034967,0.031028,0.038132,0.068796,0.020730,0.014416,0.009458,6.549680,6.378229,8.849216,5.805949,8.849216,0.001996,detectable
6,7,0.244736,0.249774,0.238038,0.076770,0.029823,0.034747,0.030228,0.037943,0.069539,0.021027,0.014733,0.010796,6.185150,5.786242,8.013504,5.746439,8.013504,0.001996,detectable
7,8,0.236307,0.220625,0.220387,0.064831,0.032645,0.035253,0.031627,0.035424,0.069258,0.020015,0.016869,0.010777,5.777156,5.335317,7.550942,4.450207,7.550942,0.001996,detectable
8,9,0.280132,0.310171,0.414177,0.087365,0.030264,0.036419,0.029980,0.039006,0.072902,0.022083,0.017483,0.012492,6.860991,7.183284,15.454466,5.594332,15.454466,0.001996,detectable
9,10,0.255655,0.213056,0.326578,0.069343,0.027868,0.037095,0.029437,0.037189,0.076108,0.017392,0.014026,0.009782,6.140711,4.937408,14.401437,5.655210,14.401437,0.001996,detectable


## Sanity Checks

In [9]:
check = perm_df.merge(
    cases_df[["case_id", "family_id", "snr", "spread_pattern", "x_distribution"]],
    on="case_id",
)

# 1) Null false-positive rate
null_mask = check["family_id"] == "Null"
n_null = int(null_mask.sum())
null_det = int((check.loc[null_mask, "classification"] == "detectable").sum())
null_fp = null_det / n_null if n_null > 0 else float("nan")
print(f"=== Null cases (n={n_null}) ===")
print(f"  Detectable (false positive): {null_det}/{n_null} = {null_fp:.1%}  (target: ~5%)")
print(f"  {check.loc[null_mask, 'classification'].value_counts().to_string()}")
print()

# 2) Detection rate by SNR
signal = check[check["family_id"] != "Null"].copy()
signal["snr_num"] = pd.to_numeric(signal["snr"], errors="coerce")
det_by_snr = (
    signal.groupby("snr_num", dropna=True)
    .apply(lambda g: pd.Series({
        "n": len(g),
        "detection_rate": (g["classification"] == "detectable").mean(),
    }))
    .reset_index()
)
print("=== Detection rate by SNR ===")
for _, row in det_by_snr.iterrows():
    bar = "█" * int(row["detection_rate"] * 40)
    print(f"  SNR={row['snr_num']:7.2f}  n={int(row['n']):>5,}  {row['detection_rate']:.1%}  {bar}")
print()

# 3) Per-family detection rate
det_by_fam = (
    signal.groupby("family_id")
    .apply(lambda g: (g["classification"] == "detectable").mean())
    .sort_values(ascending=False)
)
print("=== Detection rate by family (all SNR combined) ===")
for fid, rate in det_by_fam.items():
    bar = "█" * int(rate * 40)
    print(f"  {fid:5s}  {rate:.1%}  {bar}")
print()

# 4) Spot-check: strong linear signal
strong = (
    (check["family_id"] == "F01")
    & (check["snr"].astype(str) == "100")
    & (check["spread_pattern"] == "constant")
    & (check["x_distribution"] == "even")
)
if strong.any():
    row = check.loc[strong].iloc[0]
    print(f"=== Spot check: F01 linear, SNR=100, constant, even ===")
    print(f"  |Pearson|={row['pearson_obs']:.4f}  dcor={row['dcor_obs']:.4f}  η²={row['eta2_obs']:.4f}")
    print(f"  T={row['T_joint']:.2f}  p={row['p_value']:.6f}  → {row['classification']}")

=== Null cases (n=128) ===
  Detectable (false positive): 97/128 = 75.8%  (target: ~5%)
  classification
detectable        97
not_detectable    30
uncertain          1

=== Detection rate by SNR ===
  SNR=   0.10  n=9,504  96.5%  ██████████████████████████████████████
  SNR=   0.30  n=9,504  98.2%  ███████████████████████████████████████
  SNR=   0.50  n=9,504  98.6%  ███████████████████████████████████████
  SNR=   1.00  n=9,504  99.1%  ███████████████████████████████████████
  SNR=   2.00  n=9,504  99.3%  ███████████████████████████████████████
  SNR=   3.00  n=9,504  99.3%  ███████████████████████████████████████
  SNR=   5.00  n=9,504  99.6%  ███████████████████████████████████████
  SNR=  10.00  n=9,504  99.7%  ███████████████████████████████████████
  SNR=  20.00  n=9,504  99.8%  ███████████████████████████████████████
  SNR=  50.00  n=9,504  99.7%  ███████████████████████████████████████
  SNR= 100.00  n=9,504  99.8%  ███████████████████████████████████████
  SNR=    inf  n=9,50